# Notebook 07 — Anomaly Vectorization Comparison

This notebook compares two anomaly feature spaces to understand how vector representation affects anomaly ranking:

- **Model A**: Sparse character TF-IDF + structural features
- **Model B**: Dense TF-IDF + LSA + structural features

Goal:

- Check whether LSA compression helps or hurts anomaly ranking quality.
- Measure overlap between top anomaly sets produced by each model.
- Export side-by-side diagnostics for report discussion.

Reader guide:

- Both models use the same normalized input text; only the vector representation changes.
- Ranking outputs are saved to `data/results/`.
- Spearman rank correlation and top-k overlap are used to quantify agreement.

## Setup

Add the `src/` directory to the Python path so that project modules are importable without installing the package.

In [1]:
import sys
from pathlib import Path

project_root_path = Path.cwd().parent
if str(project_root_path / "src") not in sys.path:
    sys.path.insert(0, str(project_root_path / "src"))

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.sparse import issparse
from scipy.stats import spearmanr

from anomaly_detection import TextAnomalyDetector
from core.data_io import ArticleDataset
from exploration import attach_original_text_by_doc_id, build_anomaly_candidate_table
from preprocessing import StructuralFeatureExtractor, TextNormalizer, TextPreprocessor

## Load and normalize data


In [3]:
project_root_path = Path.cwd().parent
results_data_directory_path = project_root_path / "data" / "results"
results_data_directory_path.mkdir(parents=True, exist_ok=True)

articles_csv_path = project_root_path / "data" / "raw" / "articles.csv"
articles_data_frame = ArticleDataset(input_csv_path=articles_csv_path).load_articles()

document_ids = articles_data_frame["doc_id"].tolist()
raw_texts = articles_data_frame["text"].tolist()
normalized_bundle = TextNormalizer().normalize_for_both_tasks(raw_texts)

structural_feature_extractor = StructuralFeatureExtractor()
structural_feature_matrix = structural_feature_extractor.transform(raw_texts)
structural_feature_matrix.shape

(2164, 17)

## Model A: Sparse TF-IDF (char n-grams) + structural features

This model keeps a sparse lexical view and adds dense structural signals.


In [4]:
sparse_preprocessor = TextPreprocessor(
    vectorization_model_name="tfidf",
    max_features=30000,
    min_document_frequency=1,
    max_document_frequency=1.0,
    ngram_range=(3, 5),
    analyzer_mode="char_wb",
)

sparse_features = sparse_preprocessor.fit_transform(normalized_bundle.anomaly_texts)
if issparse(sparse_features):
    sparse_dense_features = np.asarray(sparse_features.todense(), dtype=np.float64)
else:
    sparse_dense_features = np.asarray(sparse_features, dtype=np.float64)

sparse_hybrid_features = np.hstack([sparse_dense_features, structural_feature_matrix])

sparse_detector = TextAnomalyDetector(contamination_ratio=0.02, random_seed=42)
sparse_mask, sparse_scores = sparse_detector.run_detection(sparse_hybrid_features)

sparse_table = build_anomaly_candidate_table(
    document_ids=document_ids,
    anomaly_scores=sparse_scores.tolist(),
    anomaly_mask=sparse_mask.tolist(),
)
sparse_table_with_text = attach_original_text_by_doc_id(
    anomaly_table=sparse_table,
    document_ids=document_ids,
    raw_texts=raw_texts,
)
sparse_table_with_text.to_csv(
    results_data_directory_path / "notebook_07_sparse_hybrid_anomaly_candidates.csv", index=False
)

sparse_table_with_text.head(20)

,doc_id,score,predicted_anomaly,anomaly_rank,text
0,DOC_01716,-0.390059,True,1,Individual leaders by total points (Final stan...
1,DOC_00534,-0.384093,True,2,Here are the standings after the April 6 updat...
2,DOC_01877,-0.381188,True,3,"ETHER IMPLODES 2 EARTH CORE, IS GRAVITY!!! Thi..."
3,DOC_01687,-0.381033,True,4,"Well, I'm back from Tokyo, so here are the sta..."
4,DOC_01245,-0.377375,True,5,The biggest reason why the cost of medical car...
5,DOC_01607,-0.376077,True,6,McDonnell Douglas rolls out DC-X HUNTINGTON BE...
6,DOC_00734,-0.371467,True,7,Here are the standings after game 1 of each of...
7,DOC_01037,-0.368640,True,8,The FLYERS blew a 3-0 lead over the Buffalo Sa...
8,DOC_01478,-0.366617,True,9,Subject: Re: Eugenics / ;Probably within 50 ye...
9,DOC_00847,-0.366243,True,10,Posted to the Internet by wmiler@nyx.cs.du.edu...


## Model B: Dense TF-IDF + LSA + structural features

This model uses dense semantic compression and the same structural signals.


In [5]:
lsa_preprocessor = TextPreprocessor(
    vectorization_model_name="tfidf_lsa_dense",
    max_features=30000,
    min_document_frequency=1,
    max_document_frequency=1.0,
    ngram_range=(3, 5),
    analyzer_mode="char_wb",
    dense_embedding_dimension=256,
    random_seed=42,
)

lsa_features = lsa_preprocessor.fit_transform(normalized_bundle.anomaly_texts)
lsa_dense_features = np.asarray(lsa_features, dtype=np.float64)
lsa_hybrid_features = np.hstack([lsa_dense_features, structural_feature_matrix])

lsa_detector = TextAnomalyDetector(contamination_ratio=0.02, random_seed=42)
lsa_mask, lsa_scores = lsa_detector.run_detection(lsa_hybrid_features)

lsa_table = build_anomaly_candidate_table(
    document_ids=document_ids,
    anomaly_scores=lsa_scores.tolist(),
    anomaly_mask=lsa_mask.tolist(),
)
lsa_table_with_text = attach_original_text_by_doc_id(
    anomaly_table=lsa_table,
    document_ids=document_ids,
    raw_texts=raw_texts,
)
lsa_table_with_text.to_csv(results_data_directory_path / "notebook_07_lsa_hybrid_anomaly_candidates.csv", index=False)

lsa_table_with_text.head(20)

,doc_id,score,predicted_anomaly,anomaly_rank,text
0,DOC_00098,-0.493017,True,1,GAME(S) OF 4/15 --------------- ADIRONDACK 6 C...
1,DOC_00089,-0.488938,True,2,AHL CALDER CUP PLAYOFF GAME(S) PLAYED ON 4/16 ...
2,DOC_00155,-0.486758,True,3,1993 CALDER CUP PLAYOFF SCHEDULE AND RESULTS h...
3,DOC_00746,-0.482604,True,4,Here is the price list for the week April 6 to...
4,DOC_01390,-0.482604,True,5,Here is the price list for the week April 13 t...
5,DOC_01114,-0.478307,True,6,I recently attended an allery seminar. Steroid...
6,DOC_01540,-0.476624,True,7,It is meaningless to compare one player's plus...
7,DOC_00332,-0.476374,True,8,The subject line says it all. Is it terribly d...
8,DOC_01603,-0.475963,True,9,Some recent postings remind me that I had read...
9,DOC_01687,-0.474090,True,10,"Well, I'm back from Tokyo, so here are the sta..."


## Compare top anomalies (top-20 side by side)


In [6]:
raw_text_lookup_by_doc_id = {document_id: text for document_id, text in zip(document_ids, raw_texts, strict=False)}
comparison_table = pd.DataFrame(
    {
        "rank": range(1, 21),
        "sparse_doc_id": sparse_table.head(20)["doc_id"].tolist(),
        "sparse_text": [
            str(raw_text_lookup_by_doc_id.get(document_id, ""))
            for document_id in sparse_table.head(20)["doc_id"].astype(str).tolist()
        ],
        "lsa_doc_id": lsa_table.head(20)["doc_id"].tolist(),
        "lsa_text": [
            str(raw_text_lookup_by_doc_id.get(document_id, ""))
            for document_id in lsa_table.head(20)["doc_id"].astype(str).tolist()
        ],
        "sparse_score": sparse_table.head(20)["score"].tolist(),
        "lsa_score": lsa_table.head(20)["score"].tolist(),
    }
)
comparison_table.to_csv(results_data_directory_path / "notebook_07_top20_comparison.csv", index=False)
comparison_table

,rank,sparse_doc_id,sparse_text,lsa_doc_id,lsa_text,sparse_score,lsa_score
0,1,DOC_01716,Individual leaders by total points (Final stan...,DOC_00098,GAME(S) OF 4/15 --------------- ADIRONDACK 6 C...,-0.390059,-0.493017
1,2,DOC_00534,Here are the standings after the April 6 updat...,DOC_00089,AHL CALDER CUP PLAYOFF GAME(S) PLAYED ON 4/16 ...,-0.384093,-0.488938
2,3,DOC_01877,"ETHER IMPLODES 2 EARTH CORE, IS GRAVITY!!! Thi...",DOC_00155,1993 CALDER CUP PLAYOFF SCHEDULE AND RESULTS h...,-0.381188,-0.486758
3,4,DOC_01687,"Well, I'm back from Tokyo, so here are the sta...",DOC_00746,Here is the price list for the week April 6 to...,-0.381033,-0.482604
4,5,DOC_01245,The biggest reason why the cost of medical car...,DOC_01390,Here is the price list for the week April 13 t...,-0.377375,-0.482604
5,6,DOC_01607,McDonnell Douglas rolls out DC-X HUNTINGTON BE...,DOC_01114,I recently attended an allery seminar. Steroid...,-0.376077,-0.478307
6,7,DOC_00734,Here are the standings after game 1 of each of...,DOC_01540,It is meaningless to compare one player's plus...,-0.371467,-0.476624
7,8,DOC_01037,The FLYERS blew a 3-0 lead over the Buffalo Sa...,DOC_00332,The subject line says it all. Is it terribly d...,-0.368640,-0.476374
8,9,DOC_01478,Subject: Re: Eugenics / ;Probably within 50 ye...,DOC_01603,Some recent postings remind me that I had read...,-0.366617,-0.475963
9,10,DOC_00847,Posted to the Internet by wmiler@nyx.cs.du.edu...,DOC_01687,"Well, I'm back from Tokyo, so here are the sta...",-0.366243,-0.474090


## Compare overlap diagnostics


In [7]:
def compute_top_k_overlap(
    sparse_ranked_table: pd.DataFrame,
    lsa_ranked_table: pd.DataFrame,
    top_k: int,
) -> int:
    """Calculates overlap between two ranked top-k anomaly sets."""
    sparse_top_ids = set(sparse_ranked_table.head(top_k)["doc_id"].tolist())
    lsa_top_ids = set(lsa_ranked_table.head(top_k)["doc_id"].tolist())
    return len(sparse_top_ids.intersection(lsa_top_ids))


overlap_diagnostics = pd.DataFrame(
    {
        "top_k": [10, 25, 50, 75, 100],
        "overlap_count": [
            compute_top_k_overlap(sparse_table, lsa_table, top_k_value) for top_k_value in [10, 25, 50, 75, 100]
        ],
    }
)
overlap_diagnostics.to_csv(results_data_directory_path / "notebook_07_overlap_diagnostics.csv", index=False)
overlap_diagnostics

,top_k,overlap_count
0,10,1
1,25,5
2,50,5
3,75,8
4,100,10


## Why overlap can be low


In [8]:
spearman_result = spearmanr(sparse_scores, lsa_scores)
score_rank_correlation = float(spearman_result.statistic)

summary_diagnostics = {
    "sparse_predicted_anomaly_count": int(sparse_mask.sum()),
    "lsa_predicted_anomaly_count": int(lsa_mask.sum()),
    "spearman_score_correlation": float(score_rank_correlation),
}
summary_diagnostics_data_frame = pd.DataFrame([summary_diagnostics])
summary_diagnostics_data_frame.to_csv(
    results_data_directory_path / "notebook_07_summary_diagnostics.csv",
    index=False,
)
summary_diagnostics

{'sparse_predicted_anomaly_count': 44,
 'lsa_predicted_anomaly_count': 44,
 'spearman_score_correlation': -0.15743969536053903}

## Overlap score in top-50


In [9]:
sparse_top_50 = set(sparse_table.head(50)["doc_id"].tolist())
lsa_top_50 = set(lsa_table.head(50)["doc_id"].tolist())
overlap_count = len(sparse_top_50.intersection(lsa_top_50))

pd.DataFrame([{"top_50_overlap_count": overlap_count}]).to_csv(
    results_data_directory_path / "notebook_07_top50_overlap.csv",
    index=False,
)
overlap_count

5

## Deterministic top-50 export candidate

This export uses score ranking from the dense hybrid model.
It gives exactly 50 ids for assignment-style output.


In [10]:
lsa_top_50_candidate = lsa_table.head(50).copy()
lsa_top_50_candidate["anomaly"] = lsa_top_50_candidate.index + 1

anomaly_candidate_output = lsa_top_50_candidate[["anomaly", "doc_id"]]
anomaly_candidate_output_with_text = attach_original_text_by_doc_id(
    anomaly_table=anomaly_candidate_output,
    document_ids=document_ids,
    raw_texts=raw_texts,
)
anomaly_candidate_output_with_text.to_csv(
    results_data_directory_path / "notebook_07_anomalies_candidate.csv", index=False
)

anomaly_candidate_output_with_text.head(10)

,anomaly,doc_id,text
0,1,DOC_00098,GAME(S) OF 4/15 --------------- ADIRONDACK 6 C...
1,2,DOC_00089,AHL CALDER CUP PLAYOFF GAME(S) PLAYED ON 4/16 ...
2,3,DOC_00155,1993 CALDER CUP PLAYOFF SCHEDULE AND RESULTS h...
3,4,DOC_00746,Here is the price list for the week April 6 to...
4,5,DOC_01390,Here is the price list for the week April 13 t...
5,6,DOC_01114,I recently attended an allery seminar. Steroid...
6,7,DOC_01540,It is meaningless to compare one player's plus...
7,8,DOC_00332,The subject line says it all. Is it terribly d...
8,9,DOC_01603,Some recent postings remind me that I had read...
9,10,DOC_01687,"Well, I'm back from Tokyo, so here are the sta..."


## Files produced by this notebook

- `data/results/notebook_07_sparse_hybrid_anomaly_candidates.csv`
- `data/results/notebook_07_lsa_hybrid_anomaly_candidates.csv`
- `data/results/notebook_07_top20_comparison.csv`
- `data/results/notebook_07_overlap_diagnostics.csv`
- `data/results/notebook_07_summary_diagnostics.csv`
- `data/results/notebook_07_top50_overlap.csv`
- `data/results/notebook_07_anomalies_candidate.csv`
